# Steps 
The **MICE (Multivariate Imputation by Chained Equations)** algorithm, also known as the *Iterative Imputer*, is a powerful technique for filling missing values by modeling each column as a function of the others. Here is a detailed step-by-step breakdown of how it works as explained in the video (4:38 - 14:49):

### Phase 1: Initial Imputation (Step 1)
Before starting the iterative process, the algorithm fills all missing values ($NaNs$) in the dataset with the **mean** of their respective columns. This ensures there is a complete base dataset to work with initially.

### Phase 2: Iterative Modeling (The "Chained Equations")
The algorithm then enters a loop to refine these initial guesses. It moves column by column from left to right:
1. **Re-introducing Missingness:** For the column currently being processed, the algorithm temporarily resets the previously imputed values back to *NaN*.
2. **Training a Model:** It treats this column as the target variable ($y$) and uses all other available columns as input features ($X$). It trains a regression model (e.g., *Linear Regression*) on the existing observed data.
3. **Prediction:** The trained model predicts the missing values for the target column. These new, more accurate predictions replace the initial mean-based values.
4. **Repetition:** The algorithm repeats this process for every column in the dataset, moving left to right, completing one full iteration (10:48 - 11:28).

### Phase 3: Convergence
This process is repeated over multiple iterations (11:51 - 16:30):
* **Comparison:** After each full iteration, the algorithm calculates the difference between the values predicted in the current iteration and the previous one.
* **Stabilization:** As the iterations continue, the model becomes more accurate, and the difference between successive predictions gradually decreases.
* **Stopping Criteria:** The process stops when the changes become negligible (close to zero), meaning the model has converged on a stable estimate for the missing values (17:27 - 17:41).

**Note:** Because this approach involves training multiple machine learning models repeatedly, it can be computationally intensive and slower than simpler imputation methods, requiring the full training set to be stored in memory.

# Details me 

MICE (Iterative Imputer) mein $X$ (Features) aur $Y$ (Target) ka selection process **kisi bhi fixed model ke comparison mein thoda alag aur dynamic hota hai**. Kyunki MICE mein hum *har column* ko ek baar target maan kar uske missing values predict karte hain, isliye roles baar-baar change hote hain.

### 1. $X$ aur $Y$ ka Selection (Selection Logic):
Jab aap ek specific column (man lijiye *Column A*) ke missing values ko fill kar rahe hain:
*   **$Y$ (Target):** Wo column jisme missing values hain (*Column A*). Model ko sirf isi column ke non-missing values ko seekhna hai.
*   **$X$ (Features):** Baki bache hue saare columns (*Column B, C, D...*). MICE in columns ka use karke *Column A* ka pattern samjhta hai.

### 2. Rows ko $X$ (Training) mein kab lena hai?
Ye sabse important part hai (7:54 - 9:02). Yahan **Rows ka selection** is base par hota hai ki us row mein data available hai ya nahi:

*   **Training ke liye ($X_{train}$):** Sirf wahi rows consider ki jati hain jahan par **Target Column ($Y$) ki value present (non-null) hai**. Agar kisi row mein $Y$ ki value missing hai, to us row ko training ke liye use nahi kiya ja sakta, kyunki model ke paas seekhne ke liye 'label' hi nahi hoga.
*   **Prediction ke liye ($X_{test}$):** Wo rows jahan par *Target Column ($Y$)* mein value 'Missing' (NaN) hai. Model training ke baad, inhi rows ke liye prediction generate karta hai.

### 3. Summary Table:
| Step | Column Status | Role |
| :--- | :--- | :--- |
| **Model Training** | Rows where $Y$ is NOT null | Input features ($X$) + Target ($Y$) |
| **Missing Imputation**| Rows where $Y$ IS null | Only Input features ($X$) for Prediction |

### 4. Ek Example se samjhein:
Maaniye aapke paas 3 columns hain: `Salary`, `Age`, `Experience`.
*   Agar aap `Salary` ke missing values bhar rahe hain:
    *   **$Y$:** `Salary`
    *   **$X$:** `Age` aur `Experience`
    *   **Training:** Wo saari rows jahan `Salary` di hui hai.
    *   **Prediction:** Wo rows jahan `Salary` missing hai, lekin `Age` aur `Experience` ki value maloom hai.

**Important Note:** MICE mein hum poore dataset ko iterate karte hain. Jab aap `Age` column par pahunchenge, to ab `Age` apka $Y$ ban jayega aur `Salary` (jo ab fill ho chuki hai) wo aapka $X$ ban jayegi. Isiliye isse **'Chained Equations'** kehte hain, kyunki har step par pichli predictions ka use hota hai (10:48 - 11:28).

In [201]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

In [ ]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)   
df = df.sample(5)   
df 

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [ ]:
df = df.iloc[:,0:-1]
df 

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [ ]:
df.iloc[1,0] = np.NaN 
df.iloc[3,1] = np.NaN 
df.iloc[-1,-1] = np.NaN 

In [226]:
df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


In [227]:
# Step 1 - Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [228]:
# 0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [229]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.NaN

df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


In [230]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


In [231]:
y = df1.iloc[[0,2,3,4],0]
y

21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

In [232]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

array([23.14158651])

In [233]:
df1.iloc[1,0] = 23.14

In [234]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [235]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.NaN

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


In [ ]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]] 
X

,R&D Spend,Marketing Spend
21,8.00,30.00
37,23.14,20.00
2,15.00,41.00
44,2.00,29.25


In [237]:
y = df1.iloc[[0,1,2,4],1]
y

21    15.0
37     5.0
2     10.0
44    15.0
Name: Administration, dtype: float64

In [238]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

array([11.06331285])

In [239]:
df1.iloc[3,1] = 11.06

In [240]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


In [241]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.NaN

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.14,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.06,26.0
44,2.00,15.00,NaN


In [ ]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X  

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06


In [ ]:
y = df1.iloc[0:4,-1] 
y  

21    30.0
37    20.0
2     41.0
14    26.0
Name: Marketing Spend, dtype: float64

In [244]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

array([31.56351448])

In [245]:
df1.iloc[4,-1] = 31.56

In [246]:
# After 1st Iteration
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [247]:
# Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


In [249]:
df2 = df1.copy()

df2.iloc[1,0] = np.NaN

df2

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,31.56


In [250]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

array([23.78627207])

In [252]:
df2.iloc[1,0] = 23.78

In [253]:
df2.iloc[3,1] = np.NaN
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

array([11.22020174])

In [254]:
df2.iloc[3,1] = 11.22

In [255]:
df2.iloc[4,-1] = np.NaN

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

array([38.87979054])

In [256]:
df2.iloc[4,-1] = 31.56

In [257]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [258]:
df2 - df1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.0
37,0.64,0.00,0.0
2,0.00,0.00,0.0
14,0.00,0.16,0.0
44,0.00,0.00,0.0


In [259]:
df3 = df2.copy()

df3.iloc[1,0] = np.NaN

df3

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.22,26.00
44,2.0,15.00,31.56


In [260]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

array([24.57698058])

In [261]:
df3.iloc[1,0] = 24.57

In [262]:
df3.iloc[3,1] = np.NaN
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

array([11.37282844])

In [265]:
df3.iloc[3,1] = 11.37

In [266]:
df3.iloc[4,-1] = np.NaN

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

array([45.53976417])

In [272]:
df3.iloc[4,-1] = 45.53

In [270]:
df2.iloc[3,1] = 11.22

In [273]:
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,24.57,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.37,26.00
44,2.00,15.00,45.53


In [275]:
df3 - df2

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.79,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.15,0.00
44,0.00,0.00,13.97
